# Inside vLLM's Scheduler: Continuous Batching, Chunked Prefill, and Who Waits

Companion notebook for **"Inside vLLM's Scheduler"** (GDE series #4, Nov 17, 2026).

This notebook builds a small, faithful model of vLLM's V1 scheduler and
runs the four experiments behind the post's numbers — no GPU, no vLLM
install needed. The rules it implements come from `vllm/v1/core/sched/scheduler.py`
and `vllm/config/scheduler.py` (v0.33.0):

- **One decision per step:** the scheduler outputs a dict
  `{request_id: num_tokens}` under a fixed token budget,
  `max_num_batched_tokens`. It does not distinguish prefill from decode —
  a request is prefilling or decoding only by how many of its tokens have
  been processed.
- **Decodes first:** running requests claim 1 token each; the remaining
  budget goes to prefill chunks. Chunked prefill falls out of the budget.
- **Two queues:** waiting requests are FCFS by default, or `(priority,
  arrival_time)` under `--scheduling-policy priority`. Admission is capped
  by `max_num_seqs` and by a KV-block watermark.
- **Preemption:** when a running request cannot get its next KV block,
  the newest running request is preempted (FCFS) and must recompute from
  zero. A waiting request that cannot get blocks simply waits — vLLM
  never preempts running work to admit new work.

Timing model (8B model on one H100, same calibration as GDE #3):
step time = 5 ms fixed + 40 us per token in the step; KV blocks hold
16 tokens each.


In [ ]:
"""V1-style scheduler simulator for GDE post #4 (2026-11-17).

Faithful-to-mechanics, toy-scale model of vLLM's V1 scheduler:
- one decision per step: a dict {request_id: num_tokens} under a fixed
  token budget (max_num_batched_tokens)
- running (decode) requests claim 1 token each FIRST; remaining budget
  goes to prefill chunks (chunked prefill falls out of the budget)
- waiting queue is FCFS by default, or (priority, arrival) under the
  priority policy; admission is capped by max_num_seqs and by a KV-block
  watermark; when a running request cannot get its next KV block, the
  newest running request is preempted (FCFS) and must recompute from 0.

Timing model (8B model on one H100, same calibration as GDE #3):
- step time = 5 ms fixed + 40 us per token in the step
  (prefill ~40 us/token at 40% MFU; a decode-heavy step is ~5-7 ms)
- KV blocks hold 16 tokens each (vLLM's block size)
"""
import math
import random


BLOCK = 16
STEP_BASE_S = 0.005        # 5 ms fixed cost per forward pass
STEP_PER_TOKEN_S = 40e-6   # 40 us per token


class Req:
    __slots__ = ("rid", "arrival", "prompt", "out", "computed", "ttft",
                 "first_token_time", "done_time", "admit_time",
                 "preemptions", "priority")

    def __init__(self, rid, arrival, prompt, out, priority=1):
        self.rid = rid
        self.arrival = arrival
        self.prompt = prompt
        self.out = out
        self.computed = 0
        self.ttft = None
        self.first_token_time = None
        self.done_time = None
        self.admit_time = None
        self.preemptions = 0
        self.priority = priority


def _blocks(n_tokens):
    return -(-n_tokens // BLOCK)  # ceil


def simulate(n=300, lam=3.0, budget=8192, max_seqs=256, kv_blocks=20000,
             watermark=0.01, prompt_med=800, prompt_long_frac=0.05,
             out_mean=150, seed=7, priority_frac=0.0, chunked=True,
             long_prompt_at=None, max_steps=120_000):
    """Run the step loop. Returns (metrics dict, per-request list)."""
    rng = random.Random(seed)
    reqs = []
    t = 0.0
    for i in range(n):
        t += rng.expovariate(lam)
        prompt = int(rng.lognormvariate(math.log(prompt_med), 0.9))
        if rng.random() < prompt_long_frac:
            prompt = rng.randint(8000, 12000)
        out = max(20, int(rng.gauss(out_mean, 40)))
        prio = 0 if (priority_frac and rng.random() < priority_frac) else 1
        reqs.append(Req(i, t, prompt, out, prio))
    if long_prompt_at is not None:
        # inject one very long prompt at a fixed arrival time
        reqs.append(Req(n, long_prompt_at, 12000, 200, 1))
        reqs.sort(key=lambda r: r.arrival)
        n = len(reqs)

    waiting, running, done = [], [], []
    arrivals = sorted(reqs, key=lambda r: r.arrival)
    ai = 0
    now = 0.0
    steps = 0
    preemptions = 0
    recompute_tokens = 0
    area_in_system = 0.0   # time-integral of (running + waiting)
    area_running = 0.0
    max_itl_gap = 0.0
    step_trace = []        # (time, decode_tokens, prefill_tokens, n_running)

    def free_blocks(held):
        return kv_blocks - held

    while len(done) < n and steps < max_steps:
        while ai < n and arrivals[ai].arrival <= now:
            waiting.append(arrivals[ai])
            ai += 1
        waiting.sort(key=lambda r: (r.priority, r.arrival))

        # ---- one planning pass: decodes first, then prefill chunks,
        # ---- then admission of waiting requests. Block-aware throughout:
        # ---- a running request that cannot get its next KV block preempts
        # ---- the newest other running request (FCFS); a waiting request
        # ---- that cannot get blocks for its first chunk simply waits —
        # ---- vLLM never preempts running work to admit new work. ----
        reserve = watermark * kv_blocks
        plan = {}
        tokens_used = 0

        def held_after_plan():
            return sum(_blocks(r.computed + plan.get(r.rid, 0))
                       for r in running)

        def try_allocate(r, take):
            """Fit `take` tokens for running request r into this step's
            plan, preempting the newest other running request if KV
            blocks run short, else shrinking the chunk to what fits.
            Returns tokens actually scheduled."""
            nonlocal tokens_used, preemptions, recompute_tokens
            base = r.computed + plan.get(r.rid, 0)
            while take > 0:
                need = _blocks(base + take) - _blocks(base)
                free = kv_blocks - held_after_plan()
                if need <= free:
                    plan[r.rid] = plan.get(r.rid, 0) + take
                    tokens_used += take
                    return take
                victims = [v for v in reversed(running)
                           if v is not r
                           and _blocks(v.computed + plan.get(v.rid, 0)) > 0]
                if victims:
                    victim = victims[0]
                    running.remove(victim)
                    tokens_used -= plan.pop(victim.rid, 0)
                    recompute_tokens += victim.computed
                    victim.computed = 0
                    victim.preemptions += 1
                    preemptions += 1
                    waiting.insert(0, victim)
                    continue
                # no victim can free a block: shrink to what fits
                max_take = (_blocks(base) + free) * BLOCK - base
                if max_take <= 0:
                    return 0
                take = min(take, max_take)
            return 0

        def chunk_size(r):
            remaining = r.prompt - r.computed
            if chunked:
                return min(remaining, budget - tokens_used)
            return remaining if remaining <= budget - tokens_used else 0

        # 1. decodes first: one token each
        for r in list(running):
            if r not in running:
                continue  # preempted while planning an earlier request
            if tokens_used >= budget:
                break
            if r.computed >= r.prompt:
                try_allocate(r, 1)
        # 2. running prefills fill the remaining budget
        for r in list(running):
            if r not in running:
                continue  # preempted while planning an earlier request
            if tokens_used >= budget:
                break
            if r.computed < r.prompt:
                take = chunk_size(r)
                if take > 0:
                    try_allocate(r, take)
        # 3. admit waiting requests, planning each one's first chunk now.
        #    The first chunk is sized to the remaining budget AND to the
        #    KV blocks actually free (chunked prefill needs no more than
        #    one chunk's worth of blocks to start — a 4,101-token prompt
        #    must not wait for 257 free blocks before its first token).
        for r in list(waiting):
            if tokens_used >= budget or len(running) >= max_seqs:
                break
            free_blocks_now = kv_blocks - held_after_plan() - reserve
            if free_blocks_now <= 0:
                continue
            room_tokens = free_blocks_now * BLOCK - (r.computed % BLOCK)
            take = min(r.prompt - r.computed, budget - tokens_used,
                       room_tokens)
            if not chunked:
                take = (r.prompt - r.computed
                        if r.prompt - r.computed <= budget - tokens_used
                        and r.prompt - r.computed <= room_tokens else 0)
            if take <= 0:
                continue
            if r.admit_time is None:
                r.admit_time = now
            waiting.remove(r)
            running.append(r)
            plan[r.rid] = take
            tokens_used += take

        if tokens_used == 0:
            if not running and not waiting and ai < n:
                now = arrivals[ai].arrival
                continue
            break  # nothing schedulable (pool too small for one request,
            # or every running request is stalled with no victim)

        # ---- advance one step ----
        step_s = STEP_BASE_S + STEP_PER_TOKEN_S * tokens_used
        dec_tok = sum(plan[r.rid] for r in running
                      if r.computed >= r.prompt and plan.get(r.rid))
        pre_tok = tokens_used - dec_tok
        step_trace.append((now, dec_tok, pre_tok, len(running)))
        area_in_system += (len(running) + len(waiting)) * step_s
        area_running += len(running) * step_s
        max_itl_gap = max(max_itl_gap, step_s)
        now += step_s
        steps += 1

        finished = []
        for r in list(running):
            take = plan.get(r.rid, 0)
            if take == 0:
                continue
            r.computed += take
            if r.ttft is None and r.computed >= r.prompt:
                r.ttft = now - r.arrival
                r.first_token_time = now
            if r.computed >= r.prompt + r.out:
                r.done_time = now
                finished.append(r)
        for r in finished:
            running.remove(r)
            done.append(r)

    total_time = now - reqs[0].arrival if reqs else 0.0
    ttfts = sorted(r.ttft for r in done if r.ttft is not None)
    e2es = sorted(r.done_time - r.arrival for r in done)
    itls = sorted((r.done_time - r.first_token_time) / max(1, r.out - 1)
                  for r in done if r.first_token_time is not None)

    def pct(xs, p):
        if not xs:
            return float("nan")
        i = min(len(xs) - 1, int(p * len(xs)))
        return xs[i]

    metrics = {
        "n_done": len(done),
        "steps": steps,
        "total_time_s": total_time,
        "throughput_req_s": len(done) / total_time if total_time else 0.0,
        "mean_ttft_s": sum(ttfts) / len(ttfts) if ttfts else float("nan"),
        "p50_ttft_s": pct(ttfts, 0.50),
        "p99_ttft_s": pct(ttfts, 0.99),
        "mean_itl_ms": 1000 * sum(itls) / len(itls) if itls else float("nan"),
        "p50_itl_ms": 1000 * pct(itls, 0.50),
        "p99_itl_ms": 1000 * pct(itls, 0.99),
        "mean_e2e_s": sum(e2es) / len(e2es) if e2es else float("nan"),
        "p99_e2e_s": pct(e2es, 0.99),
        "mean_in_system": area_in_system / total_time if total_time else 0.0,
        "mean_running": area_running / total_time if total_time else 0.0,
        "littles_law_L": lam * (sum(e2es) / len(e2es)) if e2es else 0.0,
        "preemptions": preemptions,
        "recompute_tokens": recompute_tokens,
        "max_step_s": max_itl_gap,
        "mean_wait_s": sum((r.admit_time - r.arrival) for r in done
                           if r.admit_time is not None) / max(1, len(done)),
    }
    return metrics, done, step_trace


def burst(budget, n_decoders=32, decoder_prompt=500, decoder_out=300,
           long_prompt=12000, chunked=True):
    """Controlled burst: N decoders already streaming + one long prompt.

    Same step rules as the full sim (decodes claim 1 token each first,
    prefill fills the remaining budget, step = 5 ms + 40 us/token).
    Returns (long-prompt TTFT in s, worst decoder step gap in s, steps).
    """
    decoders = [Req(i, 0.0, decoder_prompt, decoder_out) for i in range(n_decoders)]
    for d in decoders:
        d.computed = decoder_prompt  # already prefetched, mid-decode
    long_req = Req(n_decoders, 0.0, long_prompt, 200)
    now = 0.0
    worst_gap = 0.0
    steps = 0
    while long_req.computed < long_prompt and steps < 100_000:
        tokens = 0
        for d in decoders:
            if d.computed < d.prompt + d.out:
                tokens += 1  # one decode token each
        remaining = long_prompt - long_req.computed
        if chunked:
            take = min(remaining, budget - tokens)
        else:
            take = remaining if remaining <= budget - tokens else 0
        if take <= 0 and tokens == 0:
            break
        tokens += max(0, take)
        step_s = STEP_BASE_S + STEP_PER_TOKEN_S * tokens
        worst_gap = max(worst_gap, step_s)
        now += step_s
        steps += 1
        long_req.computed += max(0, take)
        for d in decoders:
            if d.computed < d.prompt + d.out:
                d.computed += 1
    return now, worst_gap, steps


## 1. The budget tradeoff: one long prompt joins 32 streaming chats

A 12,000-token prompt arrives while 32 chats are mid-decode. A bigger
token budget finishes the long prompt slightly faster — but every decode
in the batch waits out the whole step, so the worst gap a user feels
grows almost linearly with the budget.


In [ ]:
budgets = [512, 1024, 2048, 4096, 8192, 16384]
print(f"{'budget':>7}  {'TTFT':>6}  {'steps':>5}  {'worst decoder gap':>17}")
for b in budgets:
    ttft, gap, steps = burst(b)
    print(f"{b:7d}  {ttft:5.2f}s  {steps:5d}  {1000 * gap:16.0f} ms")

# 32x the budget buys 0.15 s of TTFT (0.64 -> 0.49) and costs 19x the gap.
ttft_512, gap_512, _ = burst(512)
ttft_16k, gap_16k, _ = burst(16384)
assert abs(ttft_512 - 0.64) < 0.02 and abs(ttft_16k - 0.49) < 0.02
assert abs(gap_512 * 1000 - 25) < 2 and abs(gap_16k * 1000 - 486) < 3
assert ttft_512 - ttft_16k < 0.2          # TTFT barely moves
assert gap_16k / gap_512 > 15            # ...while the gap explodes
print("\nOK: TTFT improves 0.15 s; worst gap grows 19x.")


## 2. Little's law: the queue is just arithmetic

Mean requests in the system (L) equals arrival rate (λ) times mean
latency (W). The scheduler cannot beat this: at a fixed arrival rate,
the only ways to shrink the queue are to cut latency or to stop
admitting work.


In [ ]:
for lam in (1.0, 2.0, 3.0, 4.0):
    m, _, _ = simulate(lam=lam, budget=8192)
    print(f"lam={lam}: measured L={m['mean_in_system']:.1f}  "
          f"lam*W={m['littles_law_L']:.1f}  mean E2E={m['mean_e2e_s']:.2f}s")
    assert abs(m['mean_in_system'] - m['littles_law_L']) < 0.15
print("OK: L == lam * W at every arrival rate.")


## 3. The preemption cliff: same traffic, smaller KV pool

Shrink the KV cache pool and nothing degrades gracefully. At 1,000
blocks (16K tokens) the run is clean; at 500 blocks the scheduler
preempts 1,195 times, p99 latency goes 1.5 s -> 54.8 s, and 60 of 150
requests never finish in 120,000 steps; at 300 blocks only 1 of 150
finishes. Every preemption throws away the victim's computed tokens
and re-pays them as prefill — the recompute work itself keeps the pool
full, which is why the collapse is a cliff, not a slope. Watch
`total_cumulative_preemption_cnt` in Prometheus: it is the first metric
to check when p99 climbs without a traffic change.


In [ ]:
for kb in (20000, 1000, 500, 300):
    m, _, _ = simulate(n=150, lam=3.0, budget=8192, kv_blocks=kb,
                       prompt_long_frac=0.0)
    print(f"KV={kb:6d} blocks ({kb * 16 / 1024:5.1f}K tok)  "
          f"preemptions={m['preemptions']:6d}  recompute={m['recompute_tokens']:>9,.0f} tok  "
          f"p99 E2E={m['p99_e2e_s']:5.1f}s  done={m['n_done']}/150")

m_big, _, _ = simulate(n=150, lam=3.0, budget=8192, kv_blocks=1000,
                       prompt_long_frac=0.0)
m_small, _, _ = simulate(n=150, lam=3.0, budget=8192, kv_blocks=500,
                         prompt_long_frac=0.0)
assert m_big['preemptions'] == 0 and m_big['n_done'] == 150
assert m_small['preemptions'] > 1000 and m_small['n_done'] < 150
print("OK: 1,000 blocks is calm; 500 blocks thrashes.")


## 4. Priority: who waits when the system is overloaded

Under the default FCFS policy every request waits in arrival order. Flip
`--scheduling-policy priority` and the 10% of requests marked high
priority jump the queue — at the cost of everyone else. At 8 requests/s
with 2,000-token median prompts (an overloaded server), priority
requests wait 2.5 s for their first token instead of 7.6 s under FCFS,
while regular requests wait 19.9 s instead of 7.6 s. Priority is a
transfer, not a speedup: it moves waiting from one class to another.
Under FCFS, preemption also takes the newest request first; under
priority it takes the lowest-priority one.


In [ ]:
m, done, _ = simulate(n=300, lam=8.0, budget=8192, priority_frac=0.10,
                       prompt_med=2000, out_mean=200)
hi = [r.ttft for r in done if r.priority == 0]
lo = [r.ttft for r in done if r.priority == 1]
m2, done2, _ = simulate(n=300, lam=8.0, budget=8192, priority_frac=0.0,
                        prompt_med=2000, out_mean=200)
fcfs = sum(r.ttft for r in done2) / len(done2)
print(f"priority TTFT mean: {sum(hi) / len(hi):.2f}s (n={len(hi)})")
print(f"regular  TTFT mean: {sum(lo) / len(lo):.2f}s (n={len(lo)})")
print(f"pure FCFS TTFT mean: {fcfs:.2f}s")
assert sum(hi) / len(hi) < fcfs < sum(lo) / len(lo)
print("OK: priority cuts its own wait ~3x and more than doubles the rest.")


## 5. Chunked prefill vs whole-prompt: the freeze, measured

The same 12,000-token prompt, two schedulers. Chunked (budget 2,048):
the prompt is sliced across 6 steps and the worst step any decoder
waits out is 87 ms. Whole-prompt (budget 16,384, chunking off): one
486 ms step freezes every decoder. This is the Sarathi-Serve result
from 2024, reproduced in miniature — and the reason vLLM V1 enables
chunked prefill by default whenever the model supports it.


In [ ]:
m, _, _ = simulate(n=60, lam=2.0, budget=2048, chunked=True,
                    long_prompt_at=5.0)
print(f"chunked B=2048:      max step={1000 * m['max_step_s']:.0f} ms")
m, _, _ = simulate(n=60, lam=2.0, budget=16384, chunked=False,
                   long_prompt_at=5.0)
print(f"whole-prompt B=16384: max step={1000 * m['max_step_s']:.0f} ms")
assert abs(1000 * m['max_step_s'] - 485) < 5
print("OK: chunking caps the freeze at ~87 ms; whole-prompt costs ~486 ms.")


## 6. Try it against real vLLM 0.33

The knobs in this notebook map one-to-one onto `vllm serve` flags.
Defaults shown are the `SchedulerConfig` defaults; `vllm serve` raises
the token budget by GPU (2,048 / 8,192 / 16,384) and
`--performance-mode throughput` doubles it.

```bash
# See the budget in action: small budget, tight ITL, slower long-prompt TTFT
vllm serve Qwen/Qwen3-8B --max-num-batched-tokens 2048 --max-num-seqs 256

# Throughput mode doubles the budget — watch p99 ITL, not throughput
vllm serve Qwen/Qwen3-8B --performance-mode throughput

# Priority scheduling: lower value = earlier (pass priority per request)
vllm serve Qwen/Qwen3-8B --scheduling-policy priority

# Cap how much prefill one long prompt can take per step while others decode
vllm serve Qwen/Qwen3-8B --long-prefill-token-threshold 1024

# The preemption counter to alert on (Prometheus)
# vllm:num_requests_preempted_total / total_cumulative_preemption_cnt
```

Honesty note: this is a toy simulator, not vLLM. The step-time model
(5 ms + 40 us/token) is calibrated to the 8B-on-H100 arithmetic in the
post, not measured on hardware; real step times depend on the model,
GPU, CUDA graphs, and prefix-cache hit rate. The qualitative results —
budget trades TTFT against worst-gap, Little's law binds, the
preemption cliff, priority as a transfer — are the scheduler's
mechanics, not artifacts of the toy. No Meta content, per standing
exclusion.
